# Clinical Feature Preparation

This notebook prepares clinical variables for the Osteosarcoma/Habitats project.

Updated preprocessing decisions:

- Translate `Sex` and `Lesion_site` into English.
- Collapse `Lesion_site` to `Femur`, `Tibia and fibula`, and `Others`.
  Baseline reports one global multi-category p value for tumor location; uni/multi logistic uses dummy-category terms with `Others` as the reference group.
- Calculate missing-data percentage for each candidate clinical feature.
- Drop variables with missing percentage `> 40%`.
- Impute remaining missing values using `KNNImputer(n_neighbors=3)`.
- Save `image_label_info_set12_clinical_variables_processed.xlsx`.
- Use the processed table for baseline table, univariate logistic regression, multivariate logistic regression, and normalized ML-ready table.

Important: tumor size features are calculated from original-space `label.nii.gz`, not resampled masks.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import itertools
import warnings 

import numpy as np
import pandas as pd
import nibabel as nb

from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import chi2, norm
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

import sys
sys.path.append('/host/d/Github/')
import Osteosarcoma.Build_lists.Build_list as Build_list

warnings.filterwarnings('ignore')


In [14]:
# ============================================================
# 2. Settings
# ============================================================

patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
split_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random0.xlsx'
patient_list_out_dir = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists'
radiomics_clinical_out_dir = '/host/d/projects/Habitats/radiomics/clinical_variables'

os.makedirs(patient_list_out_dir, exist_ok=True)
os.makedirs(radiomics_clinical_out_dir, exist_ok=True)

clinical_variables_path = os.path.join(
    patient_list_out_dir,
    'image_label_info_set12_clinical_variables.xlsx',
)
processed_clinical_variables_path = os.path.join(
    patient_list_out_dir,
    'image_label_info_set12_clinical_variables_processed.xlsx',
)
missing_report_path = os.path.join(
    patient_list_out_dir,
    'clinical_variables_missing_report.xlsx',
)
baseline_table_path = os.path.join(
    patient_list_out_dir,
    'clinical_baseline_table_prognosis.xlsx',
)
normalized_table_path = os.path.join(
    radiomics_clinical_out_dir,
    'clinical_variables_normalized.xlsx',
)
raw_model_table_path = os.path.join(
    radiomics_clinical_out_dir,
    'clinical_variables_model_matrix_raw.xlsx',
)

missing_drop_threshold = 40.0
knn_neighbors = 3

# Univariate p-value threshold for selecting variables entering multivariate logistic regression.
# This is for reporting only; later clinical ML will use all clinical variables.
univariate_p_threshold = 0.20

# Lesion-site rule based on the referenced paper's tumor-location grouping:
#   Femur
#   Tibia and fibula
#   Others
# Baseline table reports all three levels and one global multi-category p value.
# Uni/multi logistic uses dummy variables with Others as the reference group.
lesion_site_baseline_levels = ['Femur', 'Tibia and fibula', 'Others']
lesion_site_model_levels = ['Femur', 'Tibia and fibula']
lesion_site_term_name_map = {
    'Femur': 'Lesion_site_Femur',
    'Tibia and fibula': 'Lesion_site_Tibia_and_fibula',
}
lesion_site_term_display_map = {
    'Lesion_site_Femur': 'Femur vs Others',
    'Lesion_site_Tibia_and_fibula': 'Tibia and fibula vs Others',
}

def collapse_lesion_site_series(series):
    """Collapse lesion site into Femur, Tibia and fibula, or Others.

    This function is intentionally idempotent: if a previous cell already converted
    Tibia/Fibula into 'Tibia and fibula', rerunning later cells keeps that value
    instead of sending it to Others.
    """
    def _collapse_one(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        x_lower = x.lower().replace('_', ' ').replace('-', ' ')
        x_lower = ' '.join(x_lower.split())

        if x_lower == 'femur':
            return 'Femur'
        if x_lower in ['tibia', 'fibula', 'tibia and fibula', 'tibia fibula', 'tibia/fibula']:
            return 'Tibia and fibula'
        return 'Others'
    return series.map(_collapse_one)

print('patient_list_file:', patient_list_file)
print('split_file:', split_file)
print('clinical_variables_path:', clinical_variables_path)
print('processed_clinical_variables_path:', processed_clinical_variables_path)
print('missing_report_path:', missing_report_path)
print('baseline_table_path:', baseline_table_path)
print('radiomics_clinical_out_dir:', radiomics_clinical_out_dir)
print('missing_drop_threshold:', missing_drop_threshold)
print('knn_neighbors:', knn_neighbors)
print('univariate_p_threshold:', univariate_p_threshold)
print('lesion_site_baseline_levels:', lesion_site_baseline_levels)
print('lesion_site_model_levels:', lesion_site_model_levels)


patient_list_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx
split_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random0.xlsx
clinical_variables_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_clinical_variables.xlsx
processed_clinical_variables_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_clinical_variables_processed.xlsx
missing_report_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/clinical_variables_missing_report.xlsx
baseline_table_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/clinical_baseline_table_prognosis.xlsx
radiomics_clinical_out_dir: /host/d/projects/Habitats/radiomics/clinical_variables
missing_drop_threshold: 40.0
knn_neighbors: 3
univariate_p_threshold: 0.2
lesion_site_baseline_levels: ['Femur', 'Tibia and fibula', 'Others']
lesion_site_model_levels: ['Femur', 'Tibia and fibula']


In [3]:
# ============================================================
# 3. Define 330 patients
# ============================================================

build = Build_list.Build(patient_list_file)
_, patient_set_list, patient_index_list, prognosis_label_list, image_path_list, mask_path_list = build.__build__(
    label_column_name='Prognosis_label',
)

patient_df = pd.read_excel(patient_list_file, dtype={'Patient_index': str})
patient_df['Patient_set'] = patient_df['Patient_set'].astype(str)
patient_df['Patient_index'] = patient_df['Patient_index'].astype(str)

print('Number of cases:', len(patient_df))
print('Example image:', image_path_list[0])
print('Example mask :', mask_path_list[0])
print('Columns:', list(patient_df.columns))
print('Sex raw values:')
print(patient_df['Sex'].value_counts(dropna=False))
print('Lesion_site raw values:')
print(patient_df['Lesion_site'].value_counts(dropna=False))


Number of cases: 330
Example image: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz
Example mask : /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz
Columns: ['Patient_set', 'Patient_index', 'Include', 'Have_seg', 'X_shape', 'Y_shape', 'Slice_num', 'Spacing', 'Image_filepath', 'Mask_filepath', 'Medical_record_number', 'Registration_number', 'Prognosis_label', 'Pathologic_label', 'Sort_order', 'Name', 'Pathologic_fracture', 'Length', 'Width', 'Height', 'Sex', 'Side', 'Lesion_site', 'Age', 'Height_at_visit (cm)', 'Weight_at_visit (kg)', 'WBC (*10^9/L)', 'HGB (g/L)', 'PLT (*10^9/L)', 'CRP (mg/L)', 'ALP (IU/L)', 'Total_cholesterol (mmol/L)', 'Triglycerides (mmol/L)', 'LDL (mmol/L)', 'LDH (IU/L)', 'PT (S)', 'APTT (S)', 'Fibrinogen (mg/dL)', 'D-dimer (mg/L FEU)']
Sex raw values:
男    195
女    135
Name: Sex, dtype: int64
Lesion_site raw values:
股骨     175
胫骨      94
肱骨      36
腓骨      10
桡骨       5
骨盆       3
髂骨       2
尺骨       2
肩胛骨      2
腘窝       1
Na

In [4]:
# ============================================================
# 4. Helper: calculate tumor dimensions and volume from original label
# ============================================================

def calculate_label_derived_size(mask_path):
    """Calculate tumor dimensions and volume from original-space label.nii.gz.

    Professional handling notes:
    - We use the NIfTI affine to convert voxel coordinates into world RAS+ mm space.
    - AP diameter is measured along the world anterior-posterior axis, i.e. RAS Y extent.
    - Transverse diameter is measured along the world left-right axis, i.e. RAS X extent.
    - Longitudinal diameter is measured along the world superior-inferior axis, i.e. RAS Z extent.
    - Tumor volume is foreground voxel count times voxel volume from the affine determinant.

    Caveat:
    For extremity tumors, the clinical 'longitudinal' axis may mean the long axis of the involved bone.
    Here we calculate patient-anatomical SI extent from the image affine. This is reproducible and
    orientation-aware, but it may not always equal a bone-axis measurement.
    """
    out = {
        'Mask_exists': False,
        'Label_shape_x': np.nan,
        'Label_shape_y': np.nan,
        'Label_shape_z': np.nan,
        'Label_spacing_x_mm': np.nan,
        'Label_spacing_y_mm': np.nan,
        'Label_spacing_z_mm': np.nan,
        'Label_orientation': '',
        'Label_voxel_num': np.nan,
        'Tumor_AP_diameter_mm': np.nan,
        'Tumor_longitudinal_diameter_mm': np.nan,
        'Tumor_transverse_diameter_mm': np.nan,
        'Tumor_volume_mm3': np.nan,
        'Tumor_volume_cm3': np.nan,
    }

    if not isinstance(mask_path, str) or not os.path.isfile(mask_path):
        return out

    nii = nb.load(mask_path)
    data = nii.get_fdata()
    mask = data > 0
    affine = nii.affine
    zooms = nii.header.get_zooms()[:3]
    orientation = ''.join(nb.aff2axcodes(affine))

    out['Mask_exists'] = True
    out['Label_shape_x'], out['Label_shape_y'], out['Label_shape_z'] = data.shape[:3]
    out['Label_spacing_x_mm'], out['Label_spacing_y_mm'], out['Label_spacing_z_mm'] = zooms
    out['Label_orientation'] = orientation

    voxel_num = int(mask.sum())
    out['Label_voxel_num'] = voxel_num
    if voxel_num == 0:
        return out

    voxel_volume_mm3 = float(abs(np.linalg.det(affine[:3, :3])))
    tumor_volume_mm3 = voxel_num * voxel_volume_mm3
    out['Tumor_volume_mm3'] = tumor_volume_mm3
    out['Tumor_volume_cm3'] = tumor_volume_mm3 / 1000.0

    coords = np.array(np.where(mask)).T
    mins = coords.min(axis=0).astype(float)
    maxs = coords.max(axis=0).astype(float)

    # The affine maps voxel centers. Use half-voxel-expanded bbox corners to
    # measure physical edge-to-edge extent rather than center-to-center extent.
    edge_low = mins - 0.5
    edge_high = maxs + 0.5
    corner_voxels = np.array(list(itertools.product(
        [edge_low[0], edge_high[0]],
        [edge_low[1], edge_high[1]],
        [edge_low[2], edge_high[2]],
    )))
    corner_world = nb.affines.apply_affine(affine, corner_voxels)
    ras_extent = corner_world.max(axis=0) - corner_world.min(axis=0)

    # RAS+ world axes: X=left/right, Y=posterior/anterior, Z=inferior/superior.
    out['Tumor_transverse_diameter_mm'] = float(abs(ras_extent[0]))
    out['Tumor_AP_diameter_mm'] = float(abs(ras_extent[1]))
    out['Tumor_longitudinal_diameter_mm'] = float(abs(ras_extent[2]))

    return out

print(calculate_label_derived_size(mask_path_list[0]))


{'Mask_exists': True, 'Label_shape_x': 1024, 'Label_shape_y': 1024, 'Label_shape_z': 23, 'Label_spacing_x_mm': 0.37006578, 'Label_spacing_y_mm': 0.37006578, 'Label_spacing_z_mm': 11.0, 'Label_orientation': 'LPS', 'Label_voxel_num': 541863, 'Tumor_AP_diameter_mm': 122.98780633416027, 'Tumor_longitudinal_diameter_mm': 144.96823572623543, 'Tumor_transverse_diameter_mm': 116.4669886464253, 'Tumor_volume_mm3': 816281.6278881994, 'Tumor_volume_cm3': 816.2816278881994}


In [5]:
# ============================================================
# 5. Calculate label-derived tumor dimensions for all cases
# ============================================================

size_rows = []

for i in range(len(patient_df)):
    patient_set = str(patient_df.loc[i, 'Patient_set'])
    patient_index = str(patient_df.loc[i, 'Patient_index'])
    mask_path = patient_df.loc[i, 'Mask_filepath']

    print()
    print('============================================================')
    print('Processing:', patient_set, patient_index, 'i =', i)
    print('Mask:', mask_path)

    size_info = calculate_label_derived_size(mask_path)
    size_info['Patient_set'] = patient_set
    size_info['Patient_index'] = patient_index
    size_info['Mask_filepath_checked'] = mask_path
    print('  voxel num:', size_info['Label_voxel_num'])
    print('  AP/transverse/longitudinal mm:',
          size_info['Tumor_AP_diameter_mm'],
          size_info['Tumor_transverse_diameter_mm'],
          size_info['Tumor_longitudinal_diameter_mm'])
    print('  volume cm3:', size_info['Tumor_volume_cm3'])

    size_rows.append(size_info)

size_df = pd.DataFrame(size_rows)
size_df.head()



Processing: set_1 1 i = 0
Mask: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz
  voxel num: 541863
  AP/transverse/longitudinal mm: 122.98780633416027 116.4669886464253 144.96823572623543
  volume cm3: 816.2816278881994

Processing: set_1 5 i = 1
Mask: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/5/label.nii.gz
  voxel num: 128918
  AP/transverse/longitudinal mm: 79.92703217640519 79.26539745181799 89.61489220708609
  volume cm3: 137.69928971139313

Processing: set_1 7 i = 2
Mask: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/7/label.nii.gz
  voxel num: 316319
  AP/transverse/longitudinal mm: 90.6050787679851 98.94698467850685 105.82170889619738
  volume cm3: 267.14941443926966

Processing: set_1 8 i = 3
Mask: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/8/label.nii.gz
  voxel num: 39983
  AP/transverse/longitudinal mm: 40.059694807976484 41.45771585404873 48.50435836240649
  volume cm3: 30.504607392942976

Processing: set_1 11 i = 4
Mas

,Mask_exists,Label_shape_x,Label_shape_y,Label_shape_z,Label_spacing_x_mm,Label_spacing_y_mm,Label_spacing_z_mm,Label_orientation,Label_voxel_num,Tumor_AP_diameter_mm,Tumor_longitudinal_diameter_mm,Tumor_transverse_diameter_mm,Tumor_volume_mm3,Tumor_volume_cm3,Patient_set,Patient_index,Mask_filepath_checked
0,True,1024,1024,23,0.370066,0.370066,11.0,LPS,541863,122.987806,144.968236,116.466989,816281.627888,816.281628,set_1,1,/host/e/D/Data/Habitats/Jishuitan/original_dat...
1,True,512,512,20,0.390625,0.390625,7.0,LPS,128918,79.927032,89.614892,79.265397,137699.289711,137.699290,set_1,5,/host/e/D/Data/Habitats/Jishuitan/original_dat...
2,True,512,512,24,0.347349,0.347349,7.0,LPS,316319,90.605079,105.821709,98.946985,267149.414439,267.149414,set_1,7,/host/e/D/Data/Habitats/Jishuitan/original_dat...
3,True,512,512,24,0.390625,0.390625,5.0,LPS,39983,40.059695,48.504358,41.457716,30504.607393,30.504607,set_1,8,/host/e/D/Data/Habitats/Jishuitan/original_dat...
4,True,1024,1024,23,0.214844,0.214844,6.5,LPS,1207245,89.836784,89.984684,91.460815,362204.846620,362.204847,set_1,11,/host/e/D/Data/Habitats/Jishuitan/original_dat...


In [6]:
# ============================================================
# 6. Build raw clinical variable table and save
# ============================================================

# Source-to-clean-name mapping for user-defined clinical variable pool.
rename_map = {
    'Height_at_visit (cm)': 'Height_at_visit',
    'Weight_at_visit (kg)': 'Weight_at_visit',
    'WBC (*10^9/L)': 'WBC',
    'HGB (g/L)': 'HGB',
    'PLT (*10^9/L)': 'PLT',
    'CRP (mg/L)': 'CRP',
    'ALP (IU/L)': 'ALP',
    'Total_cholesterol (mmol/L)': 'Total_cholesterol',
    'Triglycerides (mmol/L)': 'Triglycerides',
    'LDL (mmol/L)': 'LDL',
    'LDH (IU/L)': 'LDH',
    'PT (S)': 'PT',
    'APTT (S)': 'APTT',
    'Fibrinogen (mg/dL)': 'Fibrinogen',
    'D-dimer (mg/L FEU)': 'D_dimer',
}

base_columns = [
    'Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath',
    'Prognosis_label', 'Pathologic_label',
    'Age', 'Sex', 'Lesion_site', 'Pathologic_fracture',
    'Height_at_visit (cm)', 'Weight_at_visit (kg)',
    'WBC (*10^9/L)', 'HGB (g/L)', 'PLT (*10^9/L)', 'CRP (mg/L)', 'ALP (IU/L)',
    'Total_cholesterol (mmol/L)', 'Triglycerides (mmol/L)', 'LDL (mmol/L)', 'LDH (IU/L)',
    'PT (S)', 'APTT (S)', 'Fibrinogen (mg/dL)', 'D-dimer (mg/L FEU)',
]

missing = [col for col in base_columns if col not in patient_df.columns]
if missing:
    raise KeyError(f'Missing expected clinical columns: {missing}')

clinical_df = patient_df[base_columns].copy().rename(columns=rename_map)
clinical_df['Patient_set'] = clinical_df['Patient_set'].astype(str)
clinical_df['Patient_index'] = clinical_df['Patient_index'].astype(str)

# Translate categorical variables into English.
sex_map = {
    '男': 'Male', '女': 'Female',
    'male': 'Male', 'female': 'Female', 'Male': 'Male', 'Female': 'Female',
    'M': 'Male', 'F': 'Female', 'm': 'Male', 'f': 'Female',
}
lesion_site_map = {
    '股骨': 'Femur',
    '胫骨': 'Tibia',
    '肱骨': 'Humerus',
    '腓骨': 'Fibula',
    '桡骨': 'Radius',
    '骨盆': 'Pelvis',
    '髂骨': 'Ilium',
    '尺骨': 'Ulna',
    '肩胛骨': 'Scapula',
    '腘窝': 'Popliteal_fossa',
}
clinical_df['Sex'] = clinical_df['Sex'].map(lambda x: sex_map.get(str(x).strip(), str(x).strip()) if pd.notna(x) else np.nan)
clinical_df['Lesion_site'] = clinical_df['Lesion_site'].map(lambda x: lesion_site_map.get(str(x).strip(), str(x).strip()) if pd.notna(x) else np.nan)

# Collapse lesion site into Femur / Tibia and fibula / Others.
clinical_df['Lesion_site'] = collapse_lesion_site_series(clinical_df['Lesion_site'])

# Convert numeric-looking columns.
for col in clinical_df.columns:
    if col not in ['Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath', 'Sex', 'Lesion_site']:
        clinical_df[col] = pd.to_numeric(clinical_df[col], errors='coerce')

clinical_df['BMI'] = clinical_df['Weight_at_visit'] / (clinical_df['Height_at_visit'] / 100.0) ** 2

# Merge label-derived tumor size features.
size_keep_columns = [
    'Patient_set', 'Patient_index',
    'Mask_exists', 'Label_shape_x', 'Label_shape_y', 'Label_shape_z',
    'Label_spacing_x_mm', 'Label_spacing_y_mm', 'Label_spacing_z_mm', 'Label_orientation',
    'Label_voxel_num',
    'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm',
    'Tumor_volume_mm3', 'Tumor_volume_cm3',
]
size_df['Patient_set'] = size_df['Patient_set'].astype(str)
size_df['Patient_index'] = size_df['Patient_index'].astype(str)
clinical_df = clinical_df.merge(size_df[size_keep_columns], on=['Patient_set', 'Patient_index'], how='left')

id_columns = ['Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath', 'Prognosis_label', 'Pathologic_label']
categorical_variables_all = ['Sex', 'Lesion_site', 'Pathologic_fracture']
continuous_variables_all = [
    'Age', 'Height_at_visit', 'Weight_at_visit', 'BMI',
    'WBC', 'HGB', 'PLT', 'CRP', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH',
    'PT', 'APTT', 'Fibrinogen', 'D_dimer',
    'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3',
]
extra_info_columns = [
    'Mask_exists', 'Label_shape_x', 'Label_shape_y', 'Label_shape_z',
    'Label_spacing_x_mm', 'Label_spacing_y_mm', 'Label_spacing_z_mm',
    'Label_orientation', 'Label_voxel_num', 'Tumor_volume_cm3',
]

clinical_df = clinical_df[id_columns + categorical_variables_all + continuous_variables_all + extra_info_columns]
clinical_df.to_excel(clinical_variables_path, index=False)

print('Saved raw clinical variable table:', clinical_variables_path)
print('Shape:', clinical_df.shape)
print('Sex translated values:')
print(clinical_df['Sex'].value_counts(dropna=False))
print('Lesion_site translated values:')
print(clinical_df['Lesion_site'].value_counts(dropna=False))
clinical_df.head()


Saved raw clinical variable table: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_clinical_variables.xlsx
Shape: (330, 40)
Sex translated values:
Male      195
Female    135
Name: Sex, dtype: int64
Lesion_site translated values:
Femur               175
Tibia and fibula    104
Others               51
Name: Lesion_site, dtype: int64


,Patient_set,Patient_index,Image_filepath,Mask_filepath,Prognosis_label,Pathologic_label,Sex,Lesion_site,Pathologic_fracture,Age,...,Mask_exists,Label_shape_x,Label_shape_y,Label_shape_z,Label_spacing_x_mm,Label_spacing_y_mm,Label_spacing_z_mm,Label_orientation,Label_voxel_num,Tumor_volume_cm3
0,set_1,1,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,1,Female,Others,0,61.0,...,True,1024,1024,23,0.370066,0.370066,11.0,LPS,541863,816.281628
1,set_1,5,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Female,Others,0,66.0,...,True,512,512,20,0.390625,0.390625,7.0,LPS,128918,137.699290
2,set_1,7,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Male,Tibia and fibula,0,16.0,...,True,512,512,24,0.347349,0.347349,7.0,LPS,316319,267.149414
3,set_1,8,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Male,Tibia and fibula,0,20.0,...,True,512,512,24,0.390625,0.390625,5.0,LPS,39983,30.504607
4,set_1,11,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,0,Male,Femur,0,18.0,...,True,1024,1024,23,0.214844,0.214844,6.5,LPS,1207245,362.204847


In [7]:
# ============================================================
# 7. Missingness report, drop >40%, and KNN imputation with 3 neighbors
# ============================================================

candidate_variables_all = categorical_variables_all + continuous_variables_all
missing_rows = []
for col in candidate_variables_all:
    missing_num = int(clinical_df[col].isna().sum())
    missing_percent = missing_num / clinical_df.shape[0] * 100.0
    missing_rows.append({
        'Variable': col,
        'Variable_type': 'categorical' if col in categorical_variables_all else 'continuous',
        'Missing_num': missing_num,
        'Missing_percent': missing_percent,
        'Drop_missing_gt_40_percent': missing_percent > missing_drop_threshold,
    })
missing_report_df = pd.DataFrame(missing_rows).sort_values('Missing_percent', ascending=False)
missing_report_df.to_excel(missing_report_path, index=False)

variables_to_drop_missing = missing_report_df.loc[
    missing_report_df['Drop_missing_gt_40_percent'], 'Variable'
].tolist()
variables_kept = [col for col in candidate_variables_all if col not in variables_to_drop_missing]
categorical_variables = [col for col in categorical_variables_all if col in variables_kept]
continuous_variables = [col for col in continuous_variables_all if col in variables_kept]

print('Saved missing report:', missing_report_path)
print('Variables dropped because missing percent > 40%:', variables_to_drop_missing)
print('Categorical variables kept:', categorical_variables)
print('Continuous variables kept:', continuous_variables)
missing_report_df


Saved missing report: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/clinical_variables_missing_report.xlsx
Variables dropped because missing percent > 40%: ['CRP']
Categorical variables kept: ['Sex', 'Lesion_site', 'Pathologic_fracture']
Continuous variables kept: ['Age', 'Height_at_visit', 'Weight_at_visit', 'BMI', 'WBC', 'HGB', 'PLT', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH', 'PT', 'APTT', 'Fibrinogen', 'D_dimer', 'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3']


,Variable,Variable_type,Missing_num,Missing_percent,Drop_missing_gt_40_percent
10,CRP,continuous,249,75.454545,True
18,Fibrinogen,continuous,76,23.030303,False
19,D_dimer,continuous,76,23.030303,False
17,APTT,continuous,72,21.818182,False
16,PT,continuous,72,21.818182,False
14,LDL,continuous,55,16.666667,False
15,LDH,continuous,42,12.727273,False
13,Triglycerides,continuous,38,11.515152,False
12,Total_cholesterol,continuous,37,11.212121,False
11,ALP,continuous,36,10.909091,False


In [8]:
# ============================================================
# 8. KNN imputation details
# ============================================================

# KNNImputer is numeric-only. For categorical variables, we encode categories
# as numeric codes, run KNN imputation jointly with continuous variables, then
# round categorical codes back to valid categories. This keeps the user's
# requested KNN-based imputation while returning human-readable categories.

processed_df = clinical_df[id_columns + variables_kept + extra_info_columns].copy()

categorical_code_maps = {}
categorical_inverse_maps = {}
encoded_for_impute = pd.DataFrame(index=processed_df.index)

for col in variables_kept:
    if col in categorical_variables:
        s = processed_df[col]
        categories = sorted([str(v) for v in s.dropna().unique()])
        code_map = {cat: idx for idx, cat in enumerate(categories)}
        inv_map = {idx: cat for cat, idx in code_map.items()}
        categorical_code_maps[col] = code_map
        categorical_inverse_maps[col] = inv_map
        encoded_for_impute[col] = s.map(lambda x: code_map.get(str(x), np.nan) if pd.notna(x) else np.nan).astype(float)
        print(col, 'category code map:', code_map)
    else:
        encoded_for_impute[col] = pd.to_numeric(processed_df[col], errors='coerce')

print('Running KNNImputer with n_neighbors =', knn_neighbors)
imputer = KNNImputer(n_neighbors=knn_neighbors, weights='uniform')
imputed_array = imputer.fit_transform(encoded_for_impute)
imputed_df = pd.DataFrame(imputed_array, columns=encoded_for_impute.columns, index=encoded_for_impute.index)

for col in variables_kept:
    if col in categorical_variables:
        valid_codes = sorted(categorical_inverse_maps[col].keys())
        min_code = min(valid_codes)
        max_code = max(valid_codes)
        rounded_codes = np.rint(imputed_df[col]).astype(int)
        rounded_codes = np.clip(rounded_codes, min_code, max_code)
        processed_df[col] = [categorical_inverse_maps[col][int(v)] for v in rounded_codes]
    else:
        processed_df[col] = imputed_df[col].astype(float)

# Enforce lesion-site grouping again before saving.
if 'Lesion_site' in processed_df.columns:
    processed_df['Lesion_site'] = collapse_lesion_site_series(processed_df['Lesion_site'])
    print('Processed Lesion_site after grouping:')
    print(processed_df['Lesion_site'].value_counts(dropna=False))

# Save a machine-readable note about what happened.
processed_df.to_excel(processed_clinical_variables_path, index=False)

print('Saved processed clinical variables:', processed_clinical_variables_path)
print('Remaining missing values in kept variables:', int(processed_df[variables_kept].isna().sum().sum()))
processed_df.head()


Sex category code map: {'Female': 0, 'Male': 1}
Lesion_site category code map: {'Femur': 0, 'Others': 1, 'Tibia and fibula': 2}
Pathologic_fracture category code map: {'0': 0, '1': 1}
Running KNNImputer with n_neighbors = 3
Processed Lesion_site after grouping:
Femur               175
Tibia and fibula    104
Others               51
Name: Lesion_site, dtype: int64
Saved processed clinical variables: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_clinical_variables_processed.xlsx
Remaining missing values in kept variables: 0


,Patient_set,Patient_index,Image_filepath,Mask_filepath,Prognosis_label,Pathologic_label,Sex,Lesion_site,Pathologic_fracture,Age,...,Mask_exists,Label_shape_x,Label_shape_y,Label_shape_z,Label_spacing_x_mm,Label_spacing_y_mm,Label_spacing_z_mm,Label_orientation,Label_voxel_num,Tumor_volume_cm3
0,set_1,1,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,1,Female,Others,0,61.0,...,True,1024,1024,23,0.370066,0.370066,11.0,LPS,541863,816.281628
1,set_1,5,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Female,Others,0,66.0,...,True,512,512,20,0.390625,0.390625,7.0,LPS,128918,137.699290
2,set_1,7,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Male,Tibia and fibula,0,16.0,...,True,512,512,24,0.347349,0.347349,7.0,LPS,316319,267.149414
3,set_1,8,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,Male,Tibia and fibula,0,20.0,...,True,512,512,24,0.390625,0.390625,5.0,LPS,39983,30.504607
4,set_1,11,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,0,Male,Femur,0,18.0,...,True,1024,1024,23,0.214844,0.214844,6.5,LPS,1207245,362.204847


In [9]:
# ============================================================
# 9. Merge processed table with fixed train/internal-test split for prognosis
# ============================================================

# Safety: collapse stale rare lesion-site categories if rerunning from this cell.
if 'Lesion_site' in processed_df.columns:
    processed_df['Lesion_site'] = collapse_lesion_site_series(processed_df['Lesion_site'])
    print('Lesion_site used for split merge:')
    print(processed_df['Lesion_site'].value_counts(dropna=False))

split_df = pd.read_excel(split_file, dtype={'Patient_index': str})
split_df['Patient_set'] = split_df['Patient_set'].astype(str)
split_df['Patient_index'] = split_df['Patient_index'].astype(str)

split_keep = ['Patient_set', 'Patient_index', 'split', 'fold', 'Prognosis_label']
for col in split_keep:
    if col not in split_df.columns:
        raise KeyError(f'Missing split column: {col}')

clinical_split_df = processed_df.drop(columns=['Prognosis_label']).merge(
    split_df[split_keep],
    on=['Patient_set', 'Patient_index'],
    how='left',
)

print('Merged processed clinical + split shape:', clinical_split_df.shape)
print(clinical_split_df[['split', 'fold', 'Prognosis_label']].value_counts(dropna=False).sort_index())
clinical_split_df.head()


Lesion_site used for split merge:
Femur               175
Tibia and fibula    104
Others               51
Name: Lesion_site, dtype: int64
Merged processed clinical + split shape: (330, 41)
split          fold  Prognosis_label
internal test  5     0                  69
                     1                  29
train          0     0                  33
                     1                  14
               1     0                  33
                     1                  14
               2     0                  33
                     1                  13
               3     0                  32
                     1                  14
               4     0                  32
                     1                  14
dtype: int64


,Patient_set,Patient_index,Image_filepath,Mask_filepath,Pathologic_label,Sex,Lesion_site,Pathologic_fracture,Age,Height_at_visit,...,Label_shape_z,Label_spacing_x_mm,Label_spacing_y_mm,Label_spacing_z_mm,Label_orientation,Label_voxel_num,Tumor_volume_cm3,split,fold,Prognosis_label
0,set_1,1,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,Female,Others,0,61.0,167.0,...,23,0.370066,0.370066,11.0,LPS,541863,816.281628,train,4,0
1,set_1,5,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,Female,Others,0,66.0,163.0,...,20,0.390625,0.390625,7.0,LPS,128918,137.699290,train,0,1
2,set_1,7,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,Male,Tibia and fibula,0,16.0,175.0,...,24,0.347349,0.347349,7.0,LPS,316319,267.149414,train,4,1
3,set_1,8,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,Male,Tibia and fibula,0,20.0,178.0,...,24,0.390625,0.390625,5.0,LPS,39983,30.504607,train,2,1
4,set_1,11,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,Male,Femur,0,18.0,171.0,...,23,0.214844,0.214844,6.5,LPS,1207245,362.204847,train,4,0


In [10]:
# ============================================================
# 10. Helper: baseline table summary and p values
# ============================================================

def format_continuous(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(values) == 0:
        return 'NA'
    median = values.median()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    mean = values.mean()
    sd = values.std(ddof=1)
    return f'{median:.3g} [{q1:.3g}, {q3:.3g}]; mean {mean:.3g} ± {sd:.3g}'


def continuous_p_value(x0, x1):
    x0 = pd.to_numeric(pd.Series(x0), errors='coerce').dropna()
    x1 = pd.to_numeric(pd.Series(x1), errors='coerce').dropna()
    if len(x0) < 2 or len(x1) < 2:
        return np.nan, 'not tested: too few values'
    normal0 = stats.shapiro(x0).pvalue > 0.05 if 3 <= len(x0) <= 5000 else False
    normal1 = stats.shapiro(x1).pvalue > 0.05 if 3 <= len(x1) <= 5000 else False
    if normal0 and normal1:
        p = stats.ttest_ind(x0, x1, equal_var=False, nan_policy='omit').pvalue
        return float(p), 'Welch t-test'
    p = stats.mannwhitneyu(x0, x1, alternative='two-sided').pvalue
    return float(p), 'Mann-Whitney U test'


def count_percent_summary(values, level):
    s = pd.Series(values).dropna().astype(str)
    if len(s) == 0:
        return '0/0 (NA%)'
    n = int((s == str(level)).sum())
    pct = n / len(s) * 100.0
    return f'{n}/{len(s)} ({pct:.1f}%)'


def categorical_global_p_value(values, group):
    """One global p value for a categorical variable across label groups.

    This is the baseline-table style used in many clinical papers: one p value
    tests whether the whole category distribution differs between label=0 and label=1.
    """
    tmp = pd.DataFrame({'value': values, 'group': group}).dropna()
    if tmp.shape[0] == 0 or tmp['group'].nunique() < 2 or tmp['value'].nunique() < 2:
        return np.nan, 'not tested: insufficient categories', np.nan

    table = pd.crosstab(tmp['value'].astype(str), tmp['group'].astype(int))
    table = table.reindex(columns=[0, 1], fill_value=0)
    chi2_stat, p_chi, dof, expected = stats.chi2_contingency(table.values)

    if table.shape == (2, 2) and np.any(expected < 5):
        _, p_fisher = stats.fisher_exact(table.values)
        return float(p_fisher), 'Fisher exact test', float(chi2_stat)

    method = 'Chi-square test'
    if np.any(expected < 5):
        method = 'Chi-square test; small expected count'
    return float(p_chi), method, float(chi2_stat)


def make_baseline_dataset_columns(dataset_name, df_group, variable, variable_type, levels=None):
    label0 = df_group[df_group['Prognosis_label'] == 0]
    label1 = df_group[df_group['Prognosis_label'] == 1]

    if variable_type == 'continuous':
        p_value, method = continuous_p_value(label0[variable], label1[variable])
        return {
            f'{dataset_name}_label0': format_continuous(label0[variable]),
            f'{dataset_name}_label1': format_continuous(label1[variable]),
            f'{dataset_name}_p_value': p_value,
            f'{dataset_name}_p_method': method,
        }

    if variable_type == 'categorical':
        p_value, method, stat = categorical_global_p_value(df_group[variable], df_group['Prognosis_label'])
        return {
            f'{dataset_name}_label0': '',
            f'{dataset_name}_label1': '',
            f'{dataset_name}_p_value': p_value,
            f'{dataset_name}_p_method': method,
            f'{dataset_name}_statistic': stat,
        }

    raise ValueError(variable_type)


In [11]:
# ============================================================
# 11. Baseline table from processed variables - paper-style wide table
# ============================================================

# Safety: make baseline robust if clinical_split_df came from an older run.
if 'Lesion_site' in clinical_split_df.columns:
    clinical_split_df['Lesion_site'] = collapse_lesion_site_series(clinical_split_df['Lesion_site'])
    print('Lesion_site used for baseline:')
    print(clinical_split_df['Lesion_site'].value_counts(dropna=False))

analysis_groups = {
    'training': clinical_split_df[clinical_split_df['fold'].isin([0, 1, 2, 3, 4])].copy(),
    'internal_test': clinical_split_df[clinical_split_df['fold'] == 5].copy(),
}

for dataset_name, df_group in analysis_groups.items():
    print('\n============================================================')
    print('Dataset:', dataset_name, 'n =', df_group.shape[0])
    print(df_group['Prognosis_label'].value_counts(dropna=False).sort_index())

baseline_rows = []

def add_categorical_variable_rows(variable, display_name, levels, level_display=None):
    if variable not in categorical_variables:
        return
    level_display = level_display or {level: str(level) for level in levels}

    # Main variable row: one global multi-category p value for each dataset.
    row = {
        'Variable': display_name,
        'Level': '',
        'Variable_type': 'categorical',
    }
    for dataset_name, df_group in analysis_groups.items():
        row.update(make_baseline_dataset_columns(dataset_name, df_group, variable, 'categorical'))
    baseline_rows.append(row)

    # Level rows: n (%) for each label group; p value left blank because the p value belongs to the whole variable.
    for level in levels:
        row = {
            'Variable': display_name,
            'Level': level_display.get(level, str(level)),
            'Variable_type': 'categorical_level',
        }
        for dataset_name, df_group in analysis_groups.items():
            label0_values = df_group.loc[df_group['Prognosis_label'] == 0, variable]
            label1_values = df_group.loc[df_group['Prognosis_label'] == 1, variable]
            row[f'{dataset_name}_label0'] = count_percent_summary(label0_values, level)
            row[f'{dataset_name}_label1'] = count_percent_summary(label1_values, level)
            row[f'{dataset_name}_p_value'] = ''
            row[f'{dataset_name}_p_method'] = ''
            row[f'{dataset_name}_statistic'] = ''
        baseline_rows.append(row)

# Categorical variables in paper-style format.
add_categorical_variable_rows(
    'Sex',
    'Sex',
    ['Female', 'Male'],
    {'Female': 'Female', 'Male': 'Male'},
)
add_categorical_variable_rows(
    'Lesion_site',
    'Tumor location',
    lesion_site_baseline_levels,
    {'Femur': 'Femur', 'Tibia and fibula': 'Tibia and fibula', 'Others': 'Others'},
)
add_categorical_variable_rows(
    'Pathologic_fracture',
    'Pathologic fracture',
    [0, 1],
    {0: 'No', 1: 'Yes'},
)

# Continuous variables: one row per variable.
for var in continuous_variables:
    row = {
        'Variable': var,
        'Level': '',
        'Variable_type': 'continuous',
    }
    for dataset_name, df_group in analysis_groups.items():
        row.update(make_baseline_dataset_columns(dataset_name, df_group, var, 'continuous'))
    baseline_rows.append(row)

baseline_table_df = pd.DataFrame(baseline_rows)

# Final paper-style column order.
ordered_cols = ['Variable', 'Level', 'Variable_type']
for dataset_name in ['training', 'internal_test']:
    ordered_cols.extend([
        f'{dataset_name}_label0',
        f'{dataset_name}_label1',
        f'{dataset_name}_p_value',
        f'{dataset_name}_p_method',
        f'{dataset_name}_statistic',
    ])
baseline_table_df = baseline_table_df[[col for col in ordered_cols if col in baseline_table_df.columns]]

baseline_table_df.to_excel(baseline_table_path, index=False)

print('Saved baseline table:', baseline_table_path)
display(baseline_table_df.head(40))


Lesion_site used for baseline:
Femur               175
Tibia and fibula    104
Others               51
Name: Lesion_site, dtype: int64

Dataset: training n = 232
0    163
1     69
Name: Prognosis_label, dtype: int64

Dataset: internal_test n = 98
0    69
1    29
Name: Prognosis_label, dtype: int64
Saved baseline table: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/clinical_baseline_table_prognosis.xlsx


,Variable,Level,Variable_type,training_label0,training_label1,training_p_value,training_p_method,training_statistic,internal_test_label0,internal_test_label1,internal_test_p_value,internal_test_p_method,internal_test_statistic
0,Sex,,categorical,,,0.222844,Chi-square test,1.485964,,,1.0,Chi-square test,0.0
1,Sex,Female,categorical_level,70/163 (42.9%),23/69 (33.3%),,,,30/69 (43.5%),12/29 (41.4%),,,
2,Sex,Male,categorical_level,93/163 (57.1%),46/69 (66.7%),,,,39/69 (56.5%),17/29 (58.6%),,,
3,Tumor location,,categorical,,,0.326196,Chi-square test,2.240517,,,0.550593,Chi-square test; small expected count,1.193518
4,Tumor location,Femur,categorical_level,78/163 (47.9%),40/69 (58.0%),,,,38/69 (55.1%),19/29 (65.5%),,,
5,Tumor location,Tibia and fibula,categorical_level,54/163 (33.1%),20/69 (29.0%),,,,22/69 (31.9%),8/29 (27.6%),,,
6,Tumor location,Others,categorical_level,31/163 (19.0%),9/69 (13.0%),,,,9/69 (13.0%),2/29 (6.9%),,,
7,Pathologic fracture,,categorical,,,0.241688,Fisher exact test,1.311968,,,0.666558,Fisher exact test,0.06468
8,Pathologic fracture,No,categorical_level,150/163 (92.0%),67/69 (97.1%),,,,64/69 (92.8%),28/29 (96.6%),,,
9,Pathologic fracture,Yes,categorical_level,13/163 (8.0%),2/69 (2.9%),,,,5/69 (7.2%),1/29 (3.4%),,,


In [12]:
# ============================================================
# 12. Helpers: logistic regression without statsmodels
# ============================================================

def prepare_design_matrix(df, variables, continuous_vars, categorical_vars, fit_info=None):
    """Prepare X matrix with one-hot encoding for logistic/ML.

    The table is already KNN-imputed, so no missing-value imputation is done here.
    Categorical variables are encoded with drop_first=True for logistic regression
    to avoid perfect collinearity with the intercept.
    """
    df = df.copy()
    X_parts = []
    term_to_variable = {}

    if fit_info is None:
        fit_info = {'dummy_columns': {}}
        learn = True
    else:
        learn = False

    for var in variables:
        if var in continuous_vars:
            s = pd.to_numeric(df[var], errors='coerce').astype(float)
            if s.isna().any():
                raise RuntimeError(f'Unexpected missing value in processed continuous variable: {var}')
            X_parts.append(pd.DataFrame({var: s}, index=df.index))
            term_to_variable[var] = var
        elif var in categorical_vars:
            s = df[var].astype(str)
            if var == 'Lesion_site':
                # Encode lesion site as dummy variables with Others as reference.
                dummies = pd.DataFrame(
                    {term_name: (s == level).astype(float) for level, term_name in lesion_site_term_name_map.items()},
                    index=df.index,
                )
            else:
                dummies = pd.get_dummies(s, prefix=var, drop_first=True, dtype=float)
            if learn:
                fit_info['dummy_columns'][var] = list(dummies.columns)
            expected_cols = fit_info['dummy_columns'][var]
            dummies = dummies.reindex(columns=expected_cols, fill_value=0.0)
            X_parts.append(dummies)
            for col in dummies.columns:
                term_to_variable[col] = var
        else:
            raise KeyError(f'Variable {var} is neither continuous nor categorical.')

    X = pd.concat(X_parts, axis=1) if len(X_parts) > 0 else pd.DataFrame(index=df.index)
    return X.astype(float), fit_info, term_to_variable


def fit_logistic_mle(X_df, y, ridge_epsilon=1e-8, maxiter=1000):
    X = np.asarray(X_df, dtype=float)
    y = np.asarray(y, dtype=float)
    X_const = np.column_stack([np.ones(X.shape[0]), X])
    names = ['Intercept'] + list(X_df.columns)

    def nll(beta):
        eta = X_const @ beta
        return np.sum(np.logaddexp(0, eta) - y * eta) + ridge_epsilon * np.sum(beta[1:] ** 2)

    result = minimize(nll, np.zeros(X_const.shape[1]), method='BFGS', options={'maxiter': maxiter})
    beta = result.x
    eta = X_const @ beta
    p = expit(eta)
    ll = np.sum(y * np.log(np.clip(p, 1e-12, 1 - 1e-12)) + (1 - y) * np.log(np.clip(1 - p, 1e-12, 1 - 1e-12)))

    W = p * (1 - p)
    information = X_const.T @ (X_const * W[:, None])
    information[1:, 1:] += np.eye(information.shape[0] - 1) * ridge_epsilon
    cov = np.linalg.pinv(information)
    se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
    z = beta / se
    p_values = 2 * (1 - norm.cdf(np.abs(z)))

    coef_df = pd.DataFrame({
        'Term': names,
        'Coef': beta,
        'SE': se,
        'Z': z,
        'P_value': p_values,
        'OR': np.exp(beta),
        'CI95_lower': np.exp(beta - 1.96 * se),
        'CI95_upper': np.exp(beta + 1.96 * se),
    })
    return {
        'success': bool(result.success),
        'message': result.message,
        'll': float(ll),
        'coef_df': coef_df,
        'n_params': X_const.shape[1],
    }


def null_log_likelihood(y):
    y = np.asarray(y, dtype=float)
    p = np.clip(y.mean(), 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))



def format_or_ci(or_value, ci_low, ci_high):
    if pd.isna(or_value) or pd.isna(ci_low) or pd.isna(ci_high):
        return ''
    return f'{or_value:.3g} ({ci_low:.3g}-{ci_high:.3g})'


def display_term_name(term):
    if term in lesion_site_term_display_map:
        return lesion_site_term_display_map[term]
    if term == 'Sex_Male':
        return 'Male vs Female'
    if term == 'Pathologic_fracture_1':
        return 'Yes vs No'
    return 'per 1 unit'


def make_logistic_paper_table(term_df, variable_p_df=None, p_col='P_value', or_label='OR_95CI'):
    """Convert term-level logistic output to a paper-style OR table."""
    if term_df is None or term_df.shape[0] == 0:
        return pd.DataFrame()

    out = term_df.copy()
    out[or_label] = [
        format_or_ci(row['OR'], row['CI95_lower'], row['CI95_upper'])
        for _, row in out.iterrows()
    ]
    out['Comparison'] = out['Term'].map(display_term_name)

    keep_cols = ['Variable', 'Term', 'Comparison', or_label, p_col, 'OR', 'CI95_lower', 'CI95_upper']
    keep_cols = [col for col in keep_cols if col in out.columns]
    out = out[keep_cols].copy()

    if variable_p_df is not None and 'Variable' in variable_p_df.columns:
        pmap = variable_p_df.set_index('Variable')['Variable_p_value'].to_dict()
        out.insert(1, 'Variable_global_p_value', out['Variable'].map(pmap))

    return out


In [13]:
# ============================================================
# 13. Univariate logistic regression on training folds 0-4
# ============================================================

# Safety: model design matrix must use grouped lesion-site values.
if 'Lesion_site' in clinical_split_df.columns:
    clinical_split_df['Lesion_site'] = collapse_lesion_site_series(clinical_split_df['Lesion_site'])

train_df = clinical_split_df[clinical_split_df['fold'].isin([0, 1, 2, 3, 4])].copy()
train_df = train_df.dropna(subset=['Prognosis_label']).copy()
train_df['Prognosis_label'] = train_df['Prognosis_label'].astype(int)

y_train = train_df['Prognosis_label'].to_numpy().astype(int)
ll_null = null_log_likelihood(y_train)
logistic_variables = categorical_variables + continuous_variables

univariate_variable_rows = []
univariate_term_rows = []

for var in logistic_variables:
    print('\n============================================================')
    print('Univariate logistic variable:', var)
    try:
        X_var, fit_info_var, term_to_variable = prepare_design_matrix(
            train_df, [var], continuous_variables, categorical_variables,
        )
        if X_var.shape[1] == 0:
            raise RuntimeError('No usable columns after encoding.')
        if np.all(X_var.nunique(dropna=False) <= 1):
            raise RuntimeError('Variable has no variation after encoding.')

        fit = fit_logistic_mle(X_var, y_train)
        lr_stat = 2 * (fit['ll'] - ll_null)
        variable_p = float(chi2.sf(lr_stat, df=X_var.shape[1]))

        print('  encoded terms:', list(X_var.columns))
        print('  model success:', fit['success'], fit['message'])
        print('  variable likelihood-ratio p:', variable_p)

        coef_df = fit['coef_df'].copy()
        coef_df = coef_df[coef_df['Term'] != 'Intercept'].copy()
        coef_df.insert(0, 'Variable', var)
        coef_df.insert(1, 'Variable_LR_P_value', variable_p)
        coef_df.insert(2, 'Fit_success', fit['success'])
        coef_df.insert(3, 'Fit_message', str(fit['message']))
        univariate_term_rows.append(coef_df)

        univariate_variable_rows.append({
            'Variable': var,
            'Variable_type': 'continuous' if var in continuous_variables else 'categorical',
            'Num_encoded_terms': X_var.shape[1],
            'LR_statistic': lr_stat,
            'Variable_p_value': variable_p,
            'Fit_success': fit['success'],
            'Fit_message': str(fit['message']),
        })
    except Exception as e:
        print('  Failed:', repr(e))
        univariate_variable_rows.append({
            'Variable': var,
            'Variable_type': 'continuous' if var in continuous_variables else 'categorical',
            'Num_encoded_terms': 0,
            'LR_statistic': np.nan,
            'Variable_p_value': np.nan,
            'Fit_success': False,
            'Fit_message': repr(e),
        })

univariate_variables_df = pd.DataFrame(univariate_variable_rows).sort_values('Variable_p_value', na_position='last')
univariate_terms_df = pd.concat(univariate_term_rows, ignore_index=True) if len(univariate_term_rows) > 0 else pd.DataFrame()

univariate_paper_table_df = make_logistic_paper_table(
    univariate_terms_df,
    variable_p_df=univariate_variables_df,
    p_col='P_value',
    or_label='Univariate_OR_95CI',
)

univariate_variables_path = os.path.join(radiomics_clinical_out_dir, 'clinical_univariate_logistic_variables.xlsx')
univariate_terms_path = os.path.join(radiomics_clinical_out_dir, 'clinical_univariate_logistic_terms.xlsx')
univariate_paper_table_path = os.path.join(radiomics_clinical_out_dir, 'clinical_univariate_logistic_paper_table.xlsx')

univariate_variables_df.to_excel(univariate_variables_path, index=False)
univariate_terms_df.to_excel(univariate_terms_path, index=False)
univariate_paper_table_df.to_excel(univariate_paper_table_path, index=False)

candidate_variables = univariate_variables_df.loc[
    univariate_variables_df['Variable_p_value'] < univariate_p_threshold,
    'Variable',
].tolist()

print('\n============================================================')
print('Saved:', univariate_variables_path)
print('Saved:', univariate_terms_path)
print('Saved:', univariate_paper_table_path)
print(f'Variables with univariate p < {univariate_p_threshold}:', len(candidate_variables))
print(candidate_variables)
display(univariate_paper_table_df.head(40))



Univariate logistic variable: Sex
  encoded terms: ['Sex_Male']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.16917933807174515

Univariate logistic variable: Lesion_site
  encoded terms: ['Lesion_site_Femur', 'Lesion_site_Tibia_and_fibula']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.3202885098116695

Univariate logistic variable: Pathologic_fracture
  encoded terms: ['Pathologic_fracture_1']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.12285105230558249

Univariate logistic variable: Age
  encoded terms: ['Age']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.7165192692635486

Univariate logistic variable: Height_at_visit
  encoded terms: ['Height_at_visit']
  model success: False Desired error not necessarily achieved due to precision loss.
  variable likelihood-ratio p: 0.3501321910590919

Univa

,Variable,Variable_global_p_value,Term,Comparison,Univariate_OR_95CI,P_value,OR,CI95_lower,CI95_upper
0,Sex,0.169179,Sex_Male,Male vs Female,1.51 (0.835-2.71),0.173342,1.505376,0.835444,2.712519
1,Lesion_site,0.320289,Lesion_site_Femur,Femur vs Others,1.77 (0.767-4.07),0.181360,1.766381,0.766924,4.068336
2,Lesion_site,0.320289,Lesion_site_Tibia_and_fibula,Tibia and fibula vs Others,1.28 (0.518-3.14),0.596797,1.275720,0.517523,3.144716
3,Pathologic_fracture,0.122851,Pathologic_fracture_1,Yes vs No,0.344 (0.0756-1.57),0.168288,0.344431,0.075608,1.569049
4,Age,0.716519,Age,per 1 unit,0.995 (0.971-1.02),0.719210,0.995476,0.971169,1.020391
5,Height_at_visit,0.350132,Height_at_visit,per 1 unit,1.01 (0.99-1.03),0.354633,1.009288,0.989722,1.029240
6,Weight_at_visit,0.606110,Weight_at_visit,per 1 unit,0.996 (0.979-1.01),0.607390,0.995637,0.979168,1.012383
7,BMI,0.177888,BMI,per 1 unit,0.954 (0.89-1.02),0.185006,0.954285,0.890487,1.022652
8,WBC,0.421773,WBC,per 1 unit,1.01 (0.988-1.03),0.416390,1.008249,0.988470,1.028424
9,HGB,0.256113,HGB,per 1 unit,0.992 (0.978-1.01),0.256920,0.992026,0.978389,1.005852


In [15]:
# ============================================================
# 14. Multivariate logistic regression using selected univariate variables
# ============================================================

if len(candidate_variables) == 0:
    print(f'No candidate variables with p < {univariate_p_threshold}. Multivariate logistic regression is skipped.')
    multivariate_terms_df = pd.DataFrame()
    independent_variables_df = pd.DataFrame()
    multivariate_paper_table_df = pd.DataFrame()
else:
    print('Candidate variables entering multivariate model:', candidate_variables)

    X_multi, multi_fit_info, multi_term_to_variable = prepare_design_matrix(
        train_df, candidate_variables, continuous_variables, categorical_variables,
    )

    n_events = int(y_train.sum())
    print('Training cases:', len(y_train))
    print('Events label=1:', n_events)
    print('Encoded multivariate terms:', X_multi.shape[1])
    print('Events per encoded term:', n_events / max(X_multi.shape[1], 1))
    if n_events / max(X_multi.shape[1], 1) < 10:
        print('Warning: events per encoded term < 10. Multivariate estimates may be unstable.')

    multi_fit = fit_logistic_mle(X_multi, y_train)
    print('Multivariate fit success:', multi_fit['success'], multi_fit['message'])

    multivariate_terms_df = multi_fit['coef_df'].copy()
    multivariate_terms_df = multivariate_terms_df[multivariate_terms_df['Term'] != 'Intercept'].copy()
    multivariate_terms_df['Variable'] = multivariate_terms_df['Term'].map(multi_term_to_variable)
    multivariate_terms_df = multivariate_terms_df[
        ['Variable', 'Term', 'Coef', 'SE', 'Z', 'P_value', 'OR', 'CI95_lower', 'CI95_upper']
    ].sort_values('P_value', na_position='last')

    # Variable-level multivariate p value by likelihood-ratio drop-one test.
    # For each selected variable, compare the full model against a model without that variable.
    multi_ll_full = multi_fit['ll']
    multivariate_variable_rows = []
    for var in candidate_variables:
        reduced_vars = [v for v in candidate_variables if v != var]
        if len(reduced_vars) == 0:
            reduced_ll = ll_null
            df_diff = X_multi.shape[1]
        else:
            X_reduced, _, _ = prepare_design_matrix(
                train_df, reduced_vars, continuous_variables, categorical_variables,
            )
            reduced_fit = fit_logistic_mle(X_reduced, y_train)
            reduced_ll = reduced_fit['ll']
            df_diff = X_multi.shape[1] - X_reduced.shape[1]
        lr_stat = 2 * (multi_ll_full - reduced_ll)
        variable_p = float(chi2.sf(lr_stat, df=max(df_diff, 1)))
        multivariate_variable_rows.append({
            'Variable': var,
            'Num_encoded_terms': int(sum(multivariate_terms_df['Variable'] == var)),
            'Adjusted_LR_statistic': lr_stat,
            'Adjusted_variable_p_value': variable_p,
        })

    multivariate_variables_df = pd.DataFrame(multivariate_variable_rows).sort_values('Adjusted_variable_p_value')

    multivariate_paper_table_df = make_logistic_paper_table(
        multivariate_terms_df,
        variable_p_df=multivariate_variables_df.rename(columns={'Adjusted_variable_p_value': 'Variable_p_value'}),
        p_col='P_value',
        or_label='Adjusted_OR_95CI',
    )

    independent_variables = sorted(multivariate_terms_df.loc[multivariate_terms_df['P_value'] < 0.05, 'Variable'].dropna().unique())
    independent_variables_df = pd.DataFrame({'Independent_variable_term_p_lt_0_05': independent_variables})
    print('Variables with any adjusted term p < 0.05:', len(independent_variables))
    print(independent_variables)

multivariate_terms_path = os.path.join(radiomics_clinical_out_dir, 'clinical_multivariate_logistic_terms.xlsx')
multivariate_variables_path = os.path.join(radiomics_clinical_out_dir, 'clinical_multivariate_logistic_variables.xlsx')
multivariate_paper_table_path = os.path.join(radiomics_clinical_out_dir, 'clinical_multivariate_logistic_paper_table.xlsx')
independent_variables_path = os.path.join(radiomics_clinical_out_dir, 'clinical_multivariate_independent_variables.xlsx')

multivariate_terms_df.to_excel(multivariate_terms_path, index=False)
if 'multivariate_variables_df' in globals():
    multivariate_variables_df.to_excel(multivariate_variables_path, index=False)
else:
    pd.DataFrame().to_excel(multivariate_variables_path, index=False)
multivariate_paper_table_df.to_excel(multivariate_paper_table_path, index=False)
independent_variables_df.to_excel(independent_variables_path, index=False)

print('Saved:', multivariate_terms_path)
print('Saved:', multivariate_variables_path)
print('Saved:', multivariate_paper_table_path)
print('Saved:', independent_variables_path)
display(multivariate_paper_table_df.head(40))


Candidate variables entering multivariate model: ['ALP', 'Tumor_transverse_diameter_mm', 'Tumor_AP_diameter_mm', 'Total_cholesterol', 'Tumor_longitudinal_diameter_mm', 'LDL', 'D_dimer', 'Triglycerides']
Training cases: 232
Events label=1: 69
Encoded multivariate terms: 8
Events per encoded term: 8.625
Multivariate fit success: False Desired error not necessarily achieved due to precision loss.
Variables with any adjusted term p < 0.05: 0
[]
Saved: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_multivariate_logistic_terms.xlsx
Saved: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_multivariate_logistic_variables.xlsx
Saved: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_multivariate_logistic_paper_table.xlsx
Saved: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_multivariate_independent_variables.xlsx


,Variable,Variable_global_p_value,Term,Comparison,Adjusted_OR_95CI,P_value,OR,CI95_lower,CI95_upper
1,ALP,0.152833,ALP,per 1 unit,1 (1-1),0.156437,1.001257,0.999519,1.002999
4,Total_cholesterol,0.509318,Total_cholesterol,per 1 unit,0.778 (0.369-1.64),0.509509,0.778062,0.369118,1.640074
8,Triglycerides,0.350328,Triglycerides,per 1 unit,0.852 (0.5-1.45),0.557829,0.852412,0.499713,1.454049
7,D_dimer,0.606447,D_dimer,per 1 unit,1.09 (0.787-1.51),0.605930,1.089218,0.787239,1.507032
5,Tumor_longitudinal_diameter_mm,0.703332,Tumor_longitudinal_diameter_mm,per 1 unit,1 (0.994-1.01),0.702607,1.001551,0.993616,1.009548
3,Tumor_AP_diameter_mm,0.707813,Tumor_AP_diameter_mm,per 1 unit,1 (0.981-1.03),0.707440,1.004645,0.980621,1.029259
6,LDL,0.359543,LDL,per 1 unit,0.957 (0.38-2.41),0.925009,0.956640,0.380082,2.407799
2,Tumor_transverse_diameter_mm,0.935022,Tumor_transverse_diameter_mm,per 1 unit,1 (0.979-1.02),0.935029,1.000908,0.979298,1.022995


In [16]:
# ============================================================
# 15. Normalize processed clinical variables and save ML-ready table
# ============================================================

# Safety: normalized ML matrix must use grouped lesion-site values.
if 'Lesion_site' in clinical_split_df.columns:
    clinical_split_df['Lesion_site'] = collapse_lesion_site_series(clinical_split_df['Lesion_site'])

# This table is intended to look like the radiomics feature tables:
# identifiers first, then normalized numeric features.
# For this preparatory table we use the already processed/imputed clinical data.
# If later you want a strictly leakage-safe clinical ML pipeline, fit encoding/scaling inside each train fold.

X_raw, ml_fit_info, ml_term_to_variable = prepare_design_matrix(
    clinical_split_df, logistic_variables, continuous_variables, categorical_variables,
)

raw_model_df = pd.concat([
    clinical_split_df[['Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath', 'Prognosis_label', 'Pathologic_label', 'split', 'fold']].reset_index(drop=True),
    X_raw.reset_index(drop=True),
], axis=1)
raw_model_df.to_excel(raw_model_table_path, index=False)

scaler = MinMaxScaler()
X_norm = pd.DataFrame(
    scaler.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index,
)
normalized_df = pd.concat([
    clinical_split_df[['Patient_set', 'Patient_index', 'Image_filepath', 'Mask_filepath', 'Prognosis_label', 'Pathologic_label', 'split', 'fold']].reset_index(drop=True),
    X_norm.reset_index(drop=True),
], axis=1)
normalized_df.to_excel(normalized_table_path, index=False)

print('Saved raw ML matrix:', raw_model_table_path)
print('Saved normalized clinical feature table:', normalized_table_path)
print('Normalized table shape:', normalized_df.shape)
print('Number of normalized clinical features:', X_norm.shape[1])
normalized_df.head()


Saved raw ML matrix: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_variables_model_matrix_raw.xlsx
Saved normalized clinical feature table: /host/d/projects/Habitats/radiomics/clinical_variables/clinical_variables_normalized.xlsx
Normalized table shape: (330, 32)
Number of normalized clinical features: 24


,Patient_set,Patient_index,Image_filepath,Mask_filepath,Prognosis_label,Pathologic_label,split,fold,Sex_Male,Lesion_site_Femur,...,LDL,LDH,PT,APTT,Fibrinogen,D_dimer,Tumor_AP_diameter_mm,Tumor_longitudinal_diameter_mm,Tumor_transverse_diameter_mm,Tumor_volume_mm3
0,set_1,1,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,1,train,4,0.0,0.0,...,0.007763,0.156334,0.696803,0.448549,0.603320,0.171512,0.581287,0.391387,0.491414,0.448871
1,set_1,5,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,train,0,0.0,0.0,...,0.008178,0.196765,0.535832,0.290237,0.384012,0.047965,0.297444,0.197822,0.273457,0.066930
2,set_1,7,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,train,4,1.0,0.0,...,0.004533,0.308625,0.607497,0.351451,0.485579,0.537791,0.367831,0.254496,0.388768,0.139791
3,set_1,8,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,1,0,train,2,1.0,0.0,...,0.006666,0.200809,0.571665,0.321636,0.295173,0.021802,0.034651,0.054063,0.051950,0.006596
4,set_1,11,/host/e/D/Data/Habitats/Jishuitan/original_dat...,/host/e/D/Data/Habitats/Jishuitan/original_dat...,0,0,train,4,1.0,1.0,...,0.002755,0.183288,0.576626,0.397098,0.322090,0.061047,0.362766,0.199115,0.344908,0.193293
